In [ ]:
import autogen
import openai
from autogen.agentchat.contrib.web_surfer import WebSurferAgent
import os


In [ ]:
api_key_gpt_4 = os.environ["AZURE_OPENAI_API_KEY"]
api_base_gpt_4 = os.environ["AZURE_API_ENDPOINT"]
gpt_4_model_name = "airt-gpt4"

openai.api_type = "azure"
openai.api_version = "2024-02-15-preview"

CONFIG_LIST = [
    {
        "model": gpt_4_model_name,
        "api_key": api_key_gpt_4,
        "base_url": api_base_gpt_4,
        "api_type": openai.api_type,
        "api_version": openai.api_version,
    },
]

llm_config = {
    "config_list": CONFIG_LIST,
    "temperature": 0,
}

In [ ]:
def _is_termination_msg(x):
    return "terminate" in x["content"].lower()
    

In [ ]:
from typing import Any, Dict
from fastcore.basics import patch


_completions_create_original = autogen.oai.client.OpenAIClient.create


# Added here so I can check the messages which are being sent to Azure
@patch  # type: ignore
def create(
    self: autogen.oai.client.OpenAIClient,
    params: Dict[str, Any],
) -> Any:
    # print(f"Messages: {params['messages']}")

    return _completions_create_original(self, params=params)

In [ ]:
from typing import Annotated
from captn.captn_agents.backend.toolboxes import Toolbox


def create_weatherman_toolbox() -> Toolbox:
    toolbox = Toolbox()

    @toolbox.add_function("""Get the weather in a city.""")
    def get_weather(
        city: Annotated[str, "The city to get the weather for."]
    ) -> str:
        return "Sunny"

    return toolbox

def create_websearch_toolbox() -> Toolbox:
    toolbox = Toolbox()

    @toolbox.add_function("""Search for information on the web.""")
    def search_web(
        query: Annotated[str, "The query to search for."]
    ) -> str:
        return "Sorry, this function is not implemented yet."

    return toolbox


In [ ]:
from typing import Union


def get_web_surfer_agent(use_real_websurfer: bool) -> Union[WebSurferAgent, autogen.AssistantAgent]:
    if use_real_websurfer:
        return WebSurferAgent(
            "web_surfer",
            llm_config=llm_config,
            summarizer_llm_config=llm_config,
            browser_config={"viewport_size": 4096, "bing_api_key": None},
            is_termination_msg=_is_termination_msg,
            human_input_mode="NEVER",
        )

    web_surfer = autogen.AssistantAgent(
        "web_surfer",
        llm_config=llm_config,
        system_message="Your job is to to pretend to be a web surfer and answer questions about the web. At the end of response, add 'Do not ask me any more questions.'",
        is_termination_msg=_is_termination_msg,
    )
    return web_surfer

    

In [ ]:
def create_team_with_toolbox(use_real_websurfer: bool) -> autogen.GroupChatManager:
    weatherman_toolbox = create_weatherman_toolbox()

    user_proxy = autogen.UserProxyAgent(
        human_input_mode="NEVER",
        name="user_proxy",
        llm_config=False,
        code_execution_config=False,
    )

    web_surfer = get_web_surfer_agent(use_real_websurfer=use_real_websurfer)

    if not use_real_websurfer:
        # Add the websearch toolbox to the web surfer agent (which is a regular AssistantAgent)
        websurfer_toolbox = create_websearch_toolbox()
        websurfer_toolbox.add_to_agent(web_surfer, user_proxy)



    weatherman = autogen.AssistantAgent(
        "weatherman",
        llm_config=llm_config,
        system_message="Your job is to respond to questions about the weather.",
        # is_termination_msg=_is_termination_msg,
    )


    weatherman_toolbox.add_to_agent(weatherman, user_proxy)


    groupchat = autogen.GroupChat(
        agents=[weatherman, web_surfer, user_proxy],
        messages=[],
    )
    manager = autogen.GroupChatManager(
        groupchat=groupchat,
        llm_config=llm_config,
    )

    return manager

In [ ]:
def create_team_without_toolbox(use_real_websurfer: bool) -> autogen.GroupChatManager:
    user_proxy = autogen.UserProxyAgent(
        human_input_mode="NEVER",
        name="user_proxy",
        llm_config=False,
        code_execution_config=False,
    )

    weatherman = autogen.AssistantAgent(
        "weatherman",
        llm_config=llm_config,
        system_message="Your job is to respond to questions about the weather.",
        # is_termination_msg=_is_termination_msg,
    )

    @user_proxy.register_for_execution()
    @weatherman.register_for_llm(description="Get the weather in a city.")
    def get_weather(
        city: Annotated[str, "The city to get the weather for."]
    ) -> str:
        return "Sunny"
    
    
    web_surfer = get_web_surfer_agent(use_real_websurfer=use_real_websurfer)
    if not use_real_websurfer:
        # Add the search_web function to the web surfer agent (which is a regular AssistantAgent)
        @user_proxy.register_for_execution()
        @web_surfer.register_for_llm(description="Search for information on the web.")
        def search_web(
            query: Annotated[str, "The query to search for."]
        ) -> str:
            return "Sorry, this function is not implemented yet."

    
    groupchat = autogen.GroupChat(
        agents=[weatherman, web_surfer, user_proxy],
        messages=[],
    )
    manager = autogen.GroupChatManager(
        groupchat=groupchat,
        llm_config=llm_config,
    )

    return manager

In [ ]:
# manager = create_team_with_toolbox(use_real_websurfer=True)
manager = create_team_without_toolbox(use_real_websurfer=True)
manager.initiate_chat(
    recipient=manager,
    message="What's the weather like in Zagreb today? And once you retrieve the weather get me some interesting facts about Zagreb from the following page https://www.infozagreb.hr/en/about-zagreb/basic-facts.",
)

In [ ]:
manager.groupchat.messages

In [ ]:
print(manager.groupchat.agents[1].name)
manager.groupchat.agents[1].chat_messages

In [ ]:
# manager.send(
#     recipient=manager,
#     message="You should also check the weather forecast for the week.",
# )

In [ ]:
# manager.send(
#     recipient=manager,
#     message="You should also check the weather forecast for the next week.",
# )